# DHCS primary-language data cleaning

This notebook is a reproducible pandas workflow for the DHCS certified-eligibles-by-primary-language extract. It retains every source row in the full output, flags suspicious values before resolving them, and never treats suppressed or absent source values as zero.

In [1]:
from pathlib import Path
import json
import pandas as pd

from clean_language_data import COUNTY_FIPS, SOURCE_COLUMNS, SPECIAL_COUNTIES

PROJECT_DIR = Path.cwd()
INPUT_PATH = PROJECT_DIR / 'data/raw/t5_eligibility_by_language_201001_202609.csv'
OUTPUT_DIR = PROJECT_DIR / 'outputs/language_cleaning'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(INPUT_PATH, dtype='string', keep_default_na=False, encoding='utf-8-sig')
assert raw.columns.tolist() == SOURCE_COLUMNS
print(f'Loaded {len(raw):,} source rows and {len(raw.columns)} columns.')

Loaded 251,535 source rows and 9 columns.


## 1. Inspect before changing anything

This profile is recorded before normalization so that missingness, categories, and anomalies are not hidden by cleaning.

In [2]:
profile = pd.DataFrame({
    'source_column': raw.columns,
    'blank_values': [int(raw[c].eq('').sum()) for c in raw.columns],
    'whitespace_changes': [int(raw[c].ne(raw[c].str.strip()).sum()) for c in raw.columns],
    'distinct_values': [int(raw[c].nunique()) for c in raw.columns],
})
display(profile)
display(raw[['County', 'Language', 'Status']].nunique().rename('distinct_values'))
print('Exact duplicate rows:', int(raw.duplicated().sum()))
print('Repeated month-county-language keys:', int(raw.duplicated(['Month of Eligibility', 'County', 'Language'], keep=False).sum()))

,source_column,blank_values,whitespace_changes,distinct_values
0,Month of Eligibility,0,0,200
1,County,0,0,60
2,Language,0,2573,34
3,Total Eligibles,94321,3,25685
4,Annotation Code,157211,0,3
5,Annotation Description,157011,0,4
6,Status,0,0,2
7,Date of Data,138139,3736,73
8,Difference,251535,0,1


County      60
Language    34
Status       2
Name: distinct_values, dtype: int64

Exact duplicate rows: 0
Repeated month-county-language keys: 0


## 2. Flag suspicious source values before resolution

The decisions are explicit: whitespace-only count cells become unknown; suppression codes remain source evidence; special geographies stay in the full table; and missing dates are not imputed.

In [3]:
blank_or_whitespace_count = raw['Total Eligibles'].str.strip().eq('')
whitespace_code_2 = (
    raw['Annotation Code'].eq('2')
    & raw['Total Eligibles'].ne('')
    & blank_or_whitespace_count
)
suspicious_values = pd.DataFrame([
    ('blank_or_whitespace_eligible_count', int(blank_or_whitespace_count.sum()), 'Retain row; count is unknown, never zero.'),
    ('code_1_suppression', int(raw['Annotation Code'].eq('1').sum()), 'Retain code and flag as suppressed.'),
    ('code_2_suppression', int(raw['Annotation Code'].eq('2').sum()), 'Retain code and flag as suppressed.'),
    ('whitespace_only_code_2_count', int(whitespace_code_2.sum()), 'Trim whitespace then retain as unknown suppressed count.'),
    ('leading_whitespace_language', int(raw['Language'].ne(raw['Language'].str.strip()).sum()), 'Trim whitespace only; do not merge categories.'),
    ('missing_or_dot_data_date', int(raw['Date of Data'].str.strip().isin({'', '.'}).sum()), 'Keep row; parsed date is missing.'),
    ('special_geography', int(raw['County'].isin(SPECIAL_COUNTIES).sum()), 'Keep full-table row; do not map it to a county polygon.'),
    ('all_blank_difference', int(raw['Difference'].eq('').sum()), 'Retain nullable field; exclude from metrics.'),
], columns=['issue', 'affected_rows', 'documented_decision'])
display(suspicious_values)
suspicious_values.to_csv(OUTPUT_DIR / 'suspicious_values.csv', index=False)

,issue,affected_rows,documented_decision
0,blank_or_whitespace_eligible_count,94324,"Retain row; count is unknown, never zero."
1,code_1_suppression,93681,Retain code and flag as suppressed.
2,code_2_suppression,643,Retain code and flag as suppressed.
3,whitespace_only_code_2_count,3,Trim whitespace then retain as unknown suppres...
4,leading_whitespace_language,2573,Trim whitespace only; do not merge categories.
5,missing_or_dot_data_date,141875,Keep row; parsed date is missing.
6,special_geography,204,Keep full-table row; do not map it to a county...
7,all_blank_difference,251535,Retain nullable field; exclude from metrics.


## 3. Normalize text and parse fields

Only surrounding whitespace and numeric display commas are removed. Language categories, status codes, and count units are preserved.

In [8]:
clean = raw.copy()
for column in clean.columns:
    clean[column] = clean[column].str.strip()

trimmed_by_column = {column: int(raw[column].ne(clean[column]).sum()) for column in clean.columns}
month_date = pd.to_datetime(clean['Month of Eligibility'], format='%Y-%m', errors='coerce')
assert month_date.notna().all(), 'Invalid eligibility month found.'

count_text = clean['Total Eligibles'].str.replace(',', '', regex=False)
eligible_count = pd.to_numeric(count_text.mask(count_text.eq('')), errors='coerce')
invalid_count = count_text.ne('') & (
    eligible_count.isna() | eligible_count.lt(0).fillna(False) | eligible_count.mod(1).ne(0).fillna(True)
)
assert not invalid_count.any(), 'Invalid nonblank count found.'
eligible_count = eligible_count.astype('Int64')

missing_date = clean['Date of Data'].isin({'', '.'})
data_date = pd.to_datetime(clean['Date of Data'].mask(missing_date), format='%d%b%Y', errors='coerce')
assert data_date[~missing_date].notna().all(), 'Invalid source data date found.'

print('Whitespace changes:', trimmed_by_column)
print('Reported count range:', int(eligible_count.min()), 'to', int(eligible_count.max()))
print('Parsed date range:', data_date.min().date(), 'to', data_date.max().date())

Whitespace changes: {'Month of Eligibility': 0, 'County': 0, 'Language': 2573, 'Total Eligibles': 3, 'Annotation Code': 0, 'Annotation Description': 0, 'Status': 0, 'Date of Data': 3736, 'Difference': 0}
Reported count range: 11 to 16051565
Parsed date range: 2020-10-19 to 2026-08-11


## 4. Preserve suppression and create the full cleaned table

A blank count is valid only when the source marks it as suppressed. This prevents the dashboard from treating privacy suppression as zero demand.

In [9]:
annotation_code = clean['Annotation Code']
is_suppressed = annotation_code.isin({'1', '2'})
suppression_description = (
    clean['Annotation Description']
    .fillna('')
    .str.strip()
    .str.startswith('Cell suppressed', na=False)
)
assert is_suppressed.eq(suppression_description).all(), (
    'Suppression code and description disagree.'
)
assert count_text.eq('').eq(is_suppressed).all(), (
    'Blank count and suppression status disagree.'
)

county_fips = clean['County'].map(COUNTY_FIPS).astype('string')
map_eligible = ~clean['County'].isin(SPECIAL_COUNTIES) & clean['Language'].ne('STATEWIDE')
assert county_fips.loc[map_eligible].notna().all(), 'A mappable county has no FIPS key.'

cleaned = pd.DataFrame({
    'month': clean['Month of Eligibility'],
    'county': clean['County'],
    'county_fips': county_fips,
    'language': clean['Language'],
    'eligible_count': eligible_count,
    'annotation_code': annotation_code,
    'annotation_description': clean['Annotation Description'],
    'status': clean['Status'],
    'data_date': data_date.dt.strftime('%Y-%m-%d').astype('string'),
    'difference': clean['Difference'].mask(clean['Difference'].eq('')),
    'is_suppressed': is_suppressed,
    'count_availability': pd.Series('reported', index=clean.index, dtype='string').mask(is_suppressed, 'suppressed'),
    'map_eligible': map_eligible,
})
assert len(cleaned) == len(raw), 'Rows must never be dropped from the full output.'
assert not cleaned.duplicated(['month', 'county', 'language']).any(), 'Duplicate cleaned key found.'
display(cleaned.head())

,month,county,county_fips,language,eligible_count,annotation_code,annotation_description,status,data_date,difference,is_suppressed,count_availability,map_eligible
0,2026-08,Alameda,06001,ASL,132,,,P,2026-08-11,<NA>,False,reported,True
1,2026-08,Alameda,06001,Arabic,2724,,,P,2026-08-11,<NA>,False,reported,True
2,2026-08,Alameda,06001,Armenian,35,,,P,2026-08-11,<NA>,False,reported,True
3,2026-08,Alameda,06001,Cambodian,715,,,P,2026-08-11,<NA>,False,reported,True
4,2026-08,Alameda,06001,Cantonese,23765,,,P,2026-08-11,<NA>,False,reported,True


## 5. Create dashboard derivatives and report every transformation

The map table excludes only special geography records. Missing county rows in a selector remain absent-source rows, not imputed zeros.

In [10]:
map_fact = cleaned.loc[cleaned['map_eligible']].copy()
selector_coverage = (
    map_fact.groupby(['month', 'language'], as_index=False)
    .agg(county_count=('county_fips', 'nunique'),
         reported_county_rows=('count_availability', lambda s: int(s.eq('reported').sum())),
         suppressed_county_rows=('count_availability', lambda s: int(s.eq('suppressed').sum())))
    .assign(counties_without_source_row=lambda x: 58 - x['county_count'],
            complete_58_county_coverage=lambda x: x['county_count'].eq(58))
)
transformation_report = pd.DataFrame([
    ('raw_profile', len(raw), len(raw), 0, 'Inspected source before changing values.'),
    ('text_normalization', len(raw), len(raw), sum(trimmed_by_column.values()), 'Trimmed whitespace only; no categories merged.'),
    ('type_conversion', len(raw), len(cleaned), int(raw['Total Eligibles'].str.contains(',', regex=False).sum()), 'Removed display commas only; count units unchanged.'),
    ('suppression_handling', len(cleaned), len(cleaned), int(is_suppressed.sum()), 'Suppressed counts remain unknown.'),
    ('map_derivative', len(cleaned), len(map_fact), len(cleaned) - len(map_fact), 'Only special geographies excluded from map output.'),
    ('selector_coverage', len(map_fact), len(selector_coverage), int((~selector_coverage['complete_58_county_coverage']).sum()), 'Flagged incomplete county coverage.'),
], columns=['stage', 'input_rows', 'output_rows', 'affected_rows', 'what_changed'])
display(transformation_report)
cleaned.to_csv(OUTPUT_DIR / 'cleaned_language_eligibles.csv', index=False)
map_fact.to_csv(OUTPUT_DIR / 'county_language_month_map.csv', index=False)
selector_coverage.to_csv(OUTPUT_DIR / 'month_language_coverage.csv', index=False)
transformation_report.to_csv(OUTPUT_DIR / 'transformation_report.csv', index=False)

,stage,input_rows,output_rows,affected_rows,what_changed
0,raw_profile,251535,251535,0,Inspected source before changing values.
1,text_normalization,251535,251535,6312,Trimmed whitespace only; no categories merged.
2,type_conversion,251535,251535,43819,Removed display commas only; count units uncha...
3,suppression_handling,251535,251535,94324,Suppressed counts remain unknown.
4,map_derivative,251535,251331,204,Only special geographies excluded from map out...
5,selector_coverage,251331,6093,5521,Flagged incomplete county coverage.


## 6. Validate the result

These checks confirm that the cleaned table retains the real-world meaning of the source: monthly county primary-language certified-eligible counts, including explicitly unknown suppressed cells.

In [11]:
reported = cleaned.loc[cleaned['count_availability'].eq('reported'), 'eligible_count']
distribution = pd.cut(reported.astype('float64'), [0, 11, 50, 100, 1_000, 10_000, 100_000, float('inf')],
                      labels=['1-10', '11-49', '50-99', '100-999', '1,000-9,999', '10,000-99,999', '100,000+'], right=False)
distribution = distribution.value_counts(sort=False).rename_axis('eligible_count_range').reset_index(name='reported_row_count')
distribution['share_of_reported_rows'] = distribution['reported_row_count'] / len(reported)

assert cleaned.loc[cleaned['is_suppressed'], 'eligible_count'].isna().all()
assert cleaned.loc[~cleaned['is_suppressed'], 'eligible_count'].notna().all()
assert map_fact['county_fips'].str.fullmatch(r'06\d{3}').all()
assert not map_fact.duplicated(['month', 'county_fips', 'language']).any()
assert (selector_coverage['county_count'] + selector_coverage['counties_without_source_row']).eq(58).all()

display(distribution)
print('Rows before/after:', len(raw), len(cleaned))
print('Reported count min/max:', int(reported.min()), int(reported.max()))
print('Reported count percentiles:', reported.quantile([.25, .5, .75, .9, .99]).to_dict())
print('Suppressed counts:', int(cleaned['is_suppressed'].sum()))
print('Complete 58-county selectors:', int(selector_coverage['complete_58_county_coverage'].sum()))
distribution.to_csv(OUTPUT_DIR / 'eligible_count_distribution.csv', index=False)

,eligible_count_range,reported_row_count,share_of_reported_rows
0,1-10,0,0.000000
1,11-49,48576,0.308986
2,50-99,21346,0.135779
3,100-999,43470,0.276507
4,"1,000-9,999",24639,0.156726
5,"10,000-99,999",14193,0.090280
6,"100,000+",4987,0.031722


Rows before/after: 251535 251535
Reported count min/max: 11 16051565
Reported count percentiles: {0.25: 36.0, 0.5: 144.0, 0.75: 1433.0, 0.9: 17018.0, 0.99: 323132.29999999923}
Suppressed counts: 94324
Complete 58-county selectors: 572
